<a href="https://colab.research.google.com/github/Datahuntl/Estudo-Comparativo-de-Detec-o-de-DeepFakes/blob/main/F3Net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
import os
import numpy as np
from PIL import Image
import cv2
from google.colab import drive
import timm

In [ ]:
drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando na instância: {device}")

In [ ]:
PATH_REAL_FACES = "/content/drive/MyDrive/IC DeepFakes/DATASET/PROCESSADO/REAL"
PATH_FAKE_FACES = "/content/drive/MyDrive/IC DeepFakes/DATASET/PROCESSADO/FAKE"

In [ ]:
class FrequencyDeepfakeDataset(Dataset):
    def __init__(self, real_face_dir, fake_face_dir, transform=None):
        self.real_images = [os.path.join(real_face_dir, img) for img in os.listdir(real_face_dir) if img.lower().endswith(('jpg', 'jpeg', 'png'))]
        self.fake_images = [os.path.join(fake_face_dir, img) for img in os.listdir(fake_face_dir) if img.lower().endswith(('jpg', 'jpeg', 'png'))]
        self.all_images = self.real_images + self.fake_images
        self.labels = [0] * len(self.real_images) + [1] * len(self.fake_images)
        self.transform = transform

    def __len__(self):
        return len(self.all_images)

    def extract_frequency_spectrum(self, pil_img):
        """ Aplica FFT para extrair o mapa de altas e baixas frequências espaciais """
        # Converter para escala de cinza usando numpy
        img_np = np.array(pil_img.convert('L'))

        # Aplicar a Transformada Rápida de Fourier de 2 Dimensões (FFT 2D)
        f = np.fft.fft2(img_np)
        fshift = np.fft.fftshift(f) # Centraliza as baixas frequências

        # Calcular o espectro de magnitude (escala logarítmica para visualização)
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1)

        # Normalizar para o intervalo [0, 255]
        magnitude_spectrum = cv2.normalize(magnitude_spectrum, None, 0, 255, cv2.NORM_MINMAX)
        return Image.fromarray(magnitude_spectrum.astype(np.uint8))

    def __getitem__(self, idx):
        img_path = self.all_images[idx]
        orig_image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        # 1. Redimensionar antes do cálculo de frequência
        orig_image = orig_image.resize((256, 256))

        # 2. Extrair o canal de frequência matemática (FFT)
        freq_image = self.extract_frequency_spectrum(orig_image)

        # 3. Mesclar: Substituímos o canal Azul (B) da imagem pelo canal de frequência
        # Isso força a CNN a aprender padrões geométricos (R, G) e artefatos de frequência (B) ao mesmo tempo!
        r, g, b = orig_image.split()
        merged_image = Image.merge('RGB', (r, g, freq_image))

        if self.transform:
            merged_image = self.transform(merged_image)

        return merged_image, label

# Transformação básica de normalização
transform_f3net = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = FrequencyDeepfakeDataset(real_face_dir=PATH_REAL, fake_face_dir=PATH_FAKE, transform=transform_f3net)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ Dataset de Frequência pronto! Total de imagens: {len(full_dataset)}")

In [ ]:
# Usaremos uma ResNet-34 pré-treinada como o extrator de características mistas (Pixel + Frequência)
model = timm.create_model('resnet34', pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 1),
    nn.Sigmoid()
)
model = model.to(device)
print("✅ Modelo Backbone para análise de Frequência configurado!")

In [ ]:
import matplotlib.pyplot as plt
import time
from tqdm.auto import tqdm

CHECKPOINT_DIR = "/content/drive/MyDrive/IC DeepFakes/F3Net_Outputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "f3net_best_checkpoint.pth")
grafico_loss_path = os.path.join(CHECKPOINT_DIR, "curva_perda_f3net.png")
grafico_acc_path = os.path.join(CHECKPOINT_DIR, "curva_acuracia_f3net.png")

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
epochs = 10
start_epoch = 0
best_val_acc = 0.0

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

if os.path.exists(checkpoint_path):
    print(f"🔄 Checkpoint encontrado em: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    history = checkpoint['history']
    print(f"▶️ Resumindo da Época [{start_epoch+1}/{epochs}]")

print("🚀 Treinando modelo no domínio da frequência...\n")
for epoch in range(start_epoch, epochs):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    all_train_labels, all_train_preds = [], []
    train_bar = tqdm(train_loader, desc=f"🎬 Época {epoch+1}/{epochs} [Treino F3Net]", leave=False)

    for batch_idx, (inputs, labels) in enumerate(train_bar):
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        train_loss += current_loss * inputs.size(0)
        preds_binary = (outputs.cpu().detach().numpy() > 0.5).astype(int)
        all_train_labels.extend(labels.cpu().numpy())
        all_train_preds.extend(preds_binary)

        if batch_idx % 2 == 0:
            batch_acc = accuracy_score(labels.cpu().numpy(), preds_binary)
            train_bar.set_postfix({"Loss": f"{current_loss:.4f}", "Acc": f"{batch_acc*100:.1f}%"})

    epoch_train_loss = train_loss / len(train_loader.dataset)
    epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

    # Validação
    model.eval()
    val_loss = 0.0
    all_labels, all_preds = [], []
    val_bar = tqdm(val_loader, desc=f"🔍 Época {epoch+1}/{epochs} [Validação F3Net]", leave=False)

    with torch.no_grad():
        for inputs, labels in val_bar:
            inputs, labels_eval = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels_eval)
            val_loss += loss.item() * inputs.size(0)
            all_labels.extend(labels.numpy())
            all_preds.extend(outputs.cpu().numpy())

    epoch_val_loss = val_loss / len(val_loader.dataset)
    all_labels, all_preds = np.array(all_labels), np.array(all_preds)
    binary_preds = (all_preds > 0.5).astype(int)

    epoch_val_acc = accuracy_score(all_labels, binary_preds)
    f1 = f1_score(all_labels, binary_preds, zero_division=0)
    prec = precision_score(all_labels, binary_preds, zero_division=0)
    rec = recall_score(all_labels, binary_preds, zero_division=0)
    try: auc = roc_auc_score(all_labels, all_preds)
    except: auc = 0.5

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)

    epoch_time = time.time() - start_time
    print(f"📊 [FIM DA ÉPOCA {epoch+1}/{epochs}] Tempo: {epoch_time:.1f}s")
    print(f"   📈 Treino     -> Perda: {epoch_train_loss:.4f} | Acurácia: {epoch_train_acc*100:.2f}%")
    print(f"   📉 Validação  -> Perda: {epoch_val_loss:.4f} | Acurácia: {epoch_val_acc*100:.2f}%")
    print(f"   ✨ Métricas   -> AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}")

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        print(f"   💾 ⭐ Novo recorde de acurácia! Salvando no Drive...")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_val_acc': best_val_acc, 'history': history}, checkpoint_path)
    print("-" * 80)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_loss"])+1), history["train_loss"], label="Treino", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_loss"])+1), history["val_loss"], label="Validação", color="red")
plt.title("Evolução da Perda - F3Net Frequência")
plt.grid(True); plt.legend(); plt.savefig(grafico_loss_path, dpi=300); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_acc"])+1), history["train_acc"], label="Treino", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_acc"])+1), history["val_acc"], label="Validação", color="green")
plt.title("Evolução da Acurácia - F3Net Frequência")
plt.grid(True); plt.legend(); plt.savefig(grafico_acc_path, dpi=300); plt.show()